In [ ]:
# E = cleaned_E
from docplex.mp.model import Model
from datetime import timedelta

# --- CONSTANTS ---
HOURS = 24
MINUTES_IN_DAY = 1440
MinUtilTime = 660
MinUtilTime_2 = 150
MaxUtilTime = 800

vehicle_fixed_cost = 1000
freq_penalty = 1000

nvehicle = 150
K = range(nvehicle)
H = range(HOURS)

bigM = 1440

Terminals=['ATB','HSK'] 
Depot='X'

# frequency 
freq1 = [0, 0, 0, 5, 4, 4, 4, 6, 6, 6, 6, 5, 4, 6, 3, 5, 5, 5, 5, 5, 5, 2, 1, 0]
freq2 = [0, 0, 0, 5, 5, 4, 4, 6, 6, 6, 6, 5, 4, 6, 3, 5, 5, 5, 5, 5, 5, 2, 1, 0]

arc_data = {
    ("X", "X"): (0, 150),
    ("X", "HSK"): (10, 150),
    ("X", "ATB"): (40, 150),
    ("HSK", "HSK"): (200, 150),
    ("ATB", "ATB"): (200, 150),
    ("HSK", "ATB"): (110, 1),
    ("ATB", "HSK"): (110, 1),
    ("HSK", "X"): (10, 150),
    ("ATB", "X"): (40, 150)
}

transitions = {"ATB": "HSK", "HSK": "ATB"}
travel_time = {("ATB", "HSK"): 110, ("HSK", "ATB"): 110}

# --- NODES ---
# Initialize the list of nodes and the node ID counter
nodes = []
node_id = 1

# Add the starting node at time 0 located at the Depot
nodes.append({'id': node_id, 'time': 0, 'loc': Depot})
start_node_id = node_id  # Save the start node ID for reference later
node_id += 1  # Increment the node ID for next use

# Loop through each hour of the day (assuming H is a list of hours, e.g., range(24))
for hour in H:
    
    # Create nodes for Terminal 1 based on its frequency at the current hour
    for i in range(freq1[hour]):
        # Calculate the minute of the day for this departure
        # Spreads trips evenly within the hour
        time_val = hour * 60 + (i * 60 // max(freq1[hour], 1))
        
        # Add a node representing a trip to Terminal 1
        nodes.append({'id': node_id, 'time': time_val, 'loc': Terminals[0]})
        node_id += 1

    # Create nodes for Terminal 2 based on its frequency at the current hour
    for i in range(freq2[hour]):
        # Same time distribution logic as above
        time_val = hour * 60 + (i * 60 // max(freq2[hour], 1))
        
        # Add a node representing a trip to Terminal 2
        nodes.append({'id': node_id, 'time': time_val, 'loc': Terminals[1]})
        node_id += 1

# Add the ending node at the end of the day (assuming MINUTES_IN_DAY = 1440)
nodes.append({'id': node_id, 'time': MINUTES_IN_DAY, 'loc': Depot})
end_node_id = node_id  # Save the end node ID for later reference
node_id += 1  # Final increment of node ID (if more nodes are to be added later)


nodes.sort(key=lambda x: x['time'])
V = nodes.copy()

# --- INITIAL ARCS ---
E = []  # List to hold all arcs (edges) in the network
synthetic_id = node_id  # Initialize synthetic node ID counter from last used node_id
synthetic_nodes = []  # List to store synthetic nodes created during transitions

# --- Part 1: Generation of arcs from the depot to terminal nodes ---
for node in nodes:
    loc = node['loc']  # Location of the node
    dst_time = node['time']  # Time associated with this node

    # Only create arcs from Depot to terminal locations
    if loc in [Terminals[0], Terminals[1]]:
        cost, N_bus = arc_data[(Depot, loc)]  # Get travel time and number of buses needed from depot to this terminal

        # Ensure it's possible to reach the node from depot before its scheduled time
        if dst_time >= cost:
            # Add arc from depot (start_node) to this terminal node
            E.append({
                'src': {'id': start_node_id, 'time': 0, 'loc': Depot},
                'dst': {'id': node['id'], 'time': dst_time, 'loc': loc},
                'cost': cost,
                'N_bus': N_bus
            })

# --- Part 2: Generation of synthetic arcs between terminal nodes, followed by arcs to depot ---
for node in nodes:
    loc = node['loc']
    time = node['time']
    src_id = node['id']

    # Skip if this location is not in transitions dictionary
    # (i.e., no defined next-hop/transition location)
    if loc not in transitions:
        continue

    # Initialize for building a path of chained transitions
    total_time = 0
    curr_loc = loc
    curr_time = time
    curr_src_id = src_id

    # Keep adding transitions while total time is within MinUtilTime_2
    while total_time + travel_time[(curr_loc, transitions[curr_loc])] <= MinUtilTime_2:
        next_loc = transitions[curr_loc]  # Get next logical terminal location
        cost = travel_time[(curr_loc, next_loc)]  # Travel time to next location
        next_time = curr_time + cost  # Time of arrival at next location

        # Stop if we exceed the end of the day
        if next_time > MINUTES_IN_DAY:
            break

        # Create a new synthetic node representing this intermediate step
        dst = {'id': synthetic_id, 'time': next_time, 'loc': next_loc}
        synthetic_nodes.append(dst)

        # Add the arc (transition) from current node to the new synthetic node
        E.append({
            'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
            'dst': dst,
            'cost': cost,
            'N_bus': 1  # Presumably always requires one bus for a hop
        })

        # Update current node information to continue the chain
        curr_src_id = synthetic_id
        curr_time = next_time
        curr_loc = next_loc
        synthetic_id += 1  # Update synthetic node ID
        total_time += cost  # Accumulate total time in chain


    # After building a valid chain of transitions, if there's time left in the day,
    # connect the last synthetic (or original) node back to the depot
    if curr_time <= MINUTES_IN_DAY:
        cost_to_X, N_bus_to_X = arc_data[(curr_loc, Depot)]  # Cost and buses required to return to depot
        E.append({
            'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
            'dst': {'id': end_node_id, 'time': MINUTES_IN_DAY, 'loc': Depot},
            'cost': cost_to_X,
            'N_bus': N_bus_to_X
        })


V.extend(synthetic_nodes)

for node in V:
    node['hhmm'] = str(timedelta(minutes=node['time']))[:-3]

# --- ADD WAITING AND DEADHEADING ARCS ---


from collections import defaultdict

MAX_WAIT = 60
augmented_E = E.copy()
unique_keys = set()

# Add existing arcs to unique_keys to avoid duplication later
for e in E:
    arc_key = (
        e['src']['id'], e['src']['time'], e['src']['loc'],
        e['dst']['id'], e['dst']['time'], e['dst']['loc'],
        e['cost'], e['N_bus']
    )
    unique_keys.add(arc_key)

# Step 1: Identify trip-ending locations
trip_end_locs = set()
for e in E:
    if e['N_bus'] == 1:
        trip_end_locs.add(e['dst']['loc'])

# Step 2: Group and sort nodes by location
nodes_by_loc = defaultdict(list)
for node in V:
    nodes_by_loc[node['loc']].append(node)

for loc in nodes_by_loc:
    nodes_by_loc[loc].sort(key=lambda n: n['time'])

# Step 3: Add waiting arcs only at trip-end locations
for loc, nodes in nodes_by_loc.items():
    if loc not in trip_end_locs:
        continue  # skip locations that are not trip ends

    for i in range(len(nodes) - 1):
        src = nodes[i]
        dst = nodes[i + 1]

        time_diff = dst['time'] - src['time']
        if 0 < time_diff <= MAX_WAIT:
            arc_key = (
                src['id'], src['time'], src['loc'],
                dst['id'], dst['time'], dst['loc'],
                200, 150
            )
            if arc_key not in unique_keys:
                unique_keys.add(arc_key)
                augmented_E.append({
                    'src': src,
                    'dst': dst,
                    'cost': 200,
                    'N_bus': 150
                })


# For each destination node in the list V
for dst in V:

    # Skip if source and destination are the same node or are at the same location
    if src['id'] == dst['id'] or src['loc'] == dst['loc']:
        continue

    # Define a key for the arc based on source and destination location
    key = (src['loc'], dst['loc'])

    # Skip if this arc is not present in arc_data (i.e., no defined travel path)
    if key not in arc_data:
        continue

    # Get travel time and capacity for the arc from arc_data
    travel_t, cap = arc_data[key]

    # Check if it's possible to travel from src to dst considering the travel time
    if dst['time'] > src['time'] + travel_t:

        # Calculate time difference: how much slack time exists after traveling
        time_diff = dst['time'] - src['time'] - travel_t

        # Maintain only the best arc (smallest time_diff) per destination location
        if dst['loc'] not in best_arcs_by_dst_loc or time_diff < best_arcs_by_dst_loc[dst['loc']]['time_diff']:
            # Store the best arc to this location with its details
            best_arcs_by_dst_loc[dst['loc']] = {
                'dst': dst,
                'travel_t': travel_t,
                'cap': cap,
                'time_diff': time_diff
            }

    # Now select the single best arc among all dst_locs (least time_diff)
    # After evaluating all dst nodes for a given src, pick the best overall arc
if best_arcs_by_dst_loc:
    # Select the destination location that yields the minimum time difference
    best_dst_loc = min(best_arcs_by_dst_loc, key=lambda loc: best_arcs_by_dst_loc[loc]['time_diff'])

    # Retrieve the corresponding arc info
    best = best_arcs_by_dst_loc[best_dst_loc]

    # Create a unique identifier for the arc to avoid duplicates
    arc_key = (
        src['id'], src['time'], src['loc'],
        best['dst']['id'], best['dst']['time'], best['dst']['loc'],
        best['travel_t'], best['cap']
    )

    # If this arc hasn't already been added, add it to augmented_E
    if arc_key not in unique_keys:
        unique_keys.add(arc_key)  # Mark this arc as added
        augmented_E.append({
            'src': src,
            'dst': best['dst'],
            'cost': 300,     # Fixed cost for this kind of augmented arc
            'N_bus': 150     # Fixed number of buses assigned (likely a placeholder or large value)
        })



# Final cleaned arcs
E = augmented_E
# --- FINAL OUTPUT [unchanged] ---
for arc in E:
    print({
        'src_id': arc['src']['id'],
        'src_loc': arc['src']['loc'],
        'src_time': arc['src']['time'],
        'dst_id': arc['dst']['id'],
        'dst_loc': arc['dst']['loc'],
        'dst_time': arc['dst']['time'],
        'cost': arc['cost'],
        'N_bus': arc['N_bus']
    })
# --- MODEL ---
# Create a new optimization model for Vehicle Scheduling
mdl = Model("VehicleScheduling")

# Set the Linear Programming method in CPLEX:
# lpmethod = 3 sets the barrier (interior-point) method for solving LP relaxations
mdl.context.cplex_parameters.lpmethod = 3

# Binary decision variable:
# x[i, j, k] = 1 if vehicle k travels from node i to node j, 0 otherwise
x = mdl.binary_var_dict(
    ((e['src']['id'], e['dst']['id'], k) for e in E for k in K),
    name=Depot  # The name is misleading here — 'Depot' is likely a string or variable for naming
)

# Binary variable z[k]: indicates whether vehicle k is used (1) or not (0)
z = mdl.binary_var_dict(K, name='z')

# Continuous variable T[k]: potentially the total working time or travel time of vehicle k
# Lower bound = 0
T = mdl.continuous_var_dict(K, name='T', lb=0)


# Collect all node IDs that appear in the arcs (E), either as source or destination
node_ids_in_arcs = set()
for e in E:
    node_ids_in_arcs.add(e['src']['id'])
    node_ids_in_arcs.add(e['dst']['id'])


arrival_time = mdl.continuous_var_dict(((k, nid) for k in K for nid in node_ids_in_arcs), name='arr', lb=0, ub=MINUTES_IN_DAY)

# Objective function:
# Minimize the total travel cost across all vehicles and arcs.
# Each arc's cost is multiplied by the binary variable x[i, j, k]
mdl.minimize(
    mdl.sum(e['cost'] * x[e['src']['id'], e['dst']['id'], k] for e in E for k in K)
)


for k in K:
    # If any arc is used by vehicle k, z[k] must be 1
    mdl.add_constraint(
        mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for e in E) <= bigM * z[k]
    )

    # If vehicle k is used (z[k] = 1), at least one arc must be assigned to it
    mdl.add_constraint(
        z[k] <= mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for e in E)
    )


     # Each used vehicle must start at the depot (start_node_id)
    mdl.add_constraint(
        mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) 
                for e in E if e['src']['id'] == start_node_id) == z[k]
    )

    # Each used vehicle must end at the depot (end_node_id)
    mdl.add_constraint(
        mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) 
                for e in E if e['dst']['id'] == end_node_id) == z[k]
    )

    for node in V:
    i = node['id']  # Extract the node ID
    
    # Skip start and end depot nodes — they are not intermediate transit nodes
    if i != start_node_id and i != end_node_id:
        mdl.add_constraint(
            # The number of arcs leaving this node (outflow) for vehicle k
            mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for e in E if e['src']['id'] == i) ==
            # Must equal the number of arcs entering this node (inflow) for vehicle k
            mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for e in E if e['dst']['id'] == i)
        )


for e in E:
    # Apply constraint only to arcs that are allowed to be used by exactly one bus
    if e['N_bus'] == 1:
        mdl.add_constraint(
            # Sum of x[src_id, dst_id, k] over all vehicles k must be ≤ 1
            # This ensures that at most one vehicle is assigned to this arc
            mdl.sum(x.get((e['src']['id'], e['dst']['id'], k), 0) for k in K) <= 1
        )

for k in K:
    # Total working time T[k] is the sum of travel times on arcs with N_bus = 1
    mdl.add_constraint(
        T[k] == mdl.sum(e['cost'] * x.get((e['src']['id'], e['dst']['id'], k), 0) 
                        for e in E if e['N_bus'] == 1)
    )

    # Working time is at least a minimum if vehicle is used
    mdl.add_constraint(T[k] >= MinUtilTime * z[k])

    # Working time is at most a maximum if vehicle is used
    mdl.add_constraint(T[k] <= MaxUtilTime * z[k])

for h in H:
    # Ensure enough trips start from Terminal 0 in hour h to satisfy frequency freq1[h]
    mdl.add_constraint(
        mdl.sum(
            x[e['src']['id'], e['dst']['id'], k]
            for e in E 
            if e['N_bus'] == 1 and e['src']['loc'] == Terminals[0] and (e['src']['time'] // 60) == h
            for k in K
        ) >= freq1[h]
    )

    # Ensure enough trips start from Terminal 1 in hour h to satisfy frequency freq2[h]
    mdl.add_constraint(
        mdl.sum(
            x[e['src']['id'], e['dst']['id'], k]
            for e in E 
            if e['N_bus'] == 1 and e['src']['loc'] == Terminals[1] and (e['src']['time'] // 60) == h
            for k in K
        ) >= freq2[h]
    )

# Identify waiting and deadheading arcs
waiting_arc_ids = {(e['src']['id'], e['dst']['id']) for e in E if e['cost'] == 200}
deadheading_arc_ids = {(e['src']['id'], e['dst']['id']) for e in E if e['cost'] == 300}

# Prevent a bus from taking more than 60 minutes of waiting at a location
for k in K:
    for node in V:
        loc = node['loc']
        nid = node['id']
        relevant_arcs = [e for e in E if e['src']['id'] == nid and e['cost'] == 200]
        mdl.add_constraint(
            mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in relevant_arcs) <= 1
        )
# Add constraint: between two scheduled trips, only one deadheading arc allowed
for k in K:
    for i in node_ids_in_arcs:
        if i in [start_node_id, end_node_id]:
            continue
        incoming_scheduled = [e for e in E if e['dst']['id'] == i and e['N_bus'] == 1]
        outgoing_deadhead = [e for e in E if e['src']['id'] == i and e['N_bus'] != 1]
        outgoing_scheduled = [e for e in E if e['src']['id'] == i and e['N_bus'] == 1]
        for s_in in incoming_scheduled:
            for d_out in outgoing_deadhead:
                for s_out in outgoing_scheduled:
                    mdl.add_constraint(
                        x.get((s_in['src']['id'], s_in['dst']['id'], k), 0) +
                        x.get((d_out['src']['id'], d_out['dst']['id'], k), 0) +
                        x.get((s_out['src']['id'], s_out['dst']['id'], k), 0) <= 2
                    )



# Prevent two consecutive deadheading arcs
for k in K:
    for e1 in E:
        for e2 in E:
            if (e1['dst']['id'] == e2['src']['id'] and
                (e1['src']['id'], e1['dst']['id']) in deadheading_arc_ids and
                (e2['src']['id'], e2['dst']['id']) in deadheading_arc_ids):
                mdl.add_constraint(
                    x[e1['src']['id'], e1['dst']['id'], k] + x[e2['src']['id'], e2['dst']['id'], k] <= 1
                )
# CPLEX tuning
mdl.context.cplex_parameters.timelimit = 8000
mdl.context.cplex_parameters.mip.tolerances.mipgap = 0.05
mdl.context.cplex_parameters.threads = 8
mdl.context.cplex_parameters.mip.strategy.heuristicfreq = 1
   

solution = mdl.solve(log_output=True)

if solution:
    # Print the total cost (objective value) of the solution
    print("Objective:", solution.objective_value)

    # Identify which vehicles are used (z[k] > 0.5 indicates usage)
    used_vehicles = [k for k in K if z[k].solution_value > 0.5]
    print(f"Number of vehicles used: {len(used_vehicles)}\n")

    # Loop through each used vehicle and display its route
    for k in used_vehicles:
        # Print the total utilization time for vehicle k
        print(f"Schedule for Vehicle {k + 1}: Utilization time = {T[k].solution_value:.2f}")

        current_node = start_node_id  # Start from the depot

        # Trace the full path taken by vehicle k
        while current_node != end_node_id:
            # Find the next arc taken by this vehicle from the current node
            next_arcs = [
                e for e in E 
                if e['src']['id'] == current_node and 
                x.get((e['src']['id'], e['dst']['id'], k), 0).solution_value > 0.5
            ]

            # If no outgoing arc is found, the route is incomplete or disconnected
            if not next_arcs:
                print("  ERROR: Route incomplete or disconnected.")
                break

            arc = next_arcs[0]  # Assuming only one active outgoing arc per node
            src, dst = arc['src'], arc['dst']

            # Convert times from minutes to HH:MM format for readability
            src_h, src_m = divmod(int(src['time']), 60)
            dst_h, dst_m = divmod(int(dst['time']), 60)

            # Print the movement from source to destination with time and cost
            print(f"  ({src['loc']} at {src_h:02d}:{src_m:02d}) --> ({dst['loc']} at {dst_h:02d}:{dst_m:02d}), cost={arc['cost']}")

            current_node = dst['id']  # Move to the next node

        print()  # Add a line break between vehicle schedules

else:
    # If no feasible solution is found by the solver
    print("No feasible solution found.")

# Print the total number of variables in the model for diagnostic purposes
print(mdl.number_of_variables)
